# Meteo Data

Weather data is essential for energy data analysis — outdoor temperature drives heating demand, solar radiation affects PV production, and humidity influences cooling loads. In Switzerland, two main **open data sources** provide free weather station data:

| Provider | Description | Data Access | Resolution |
|---|---|---|---|
| **MeteoSwiss** | Federal Office of Meteorology and Climatology, official network | Station lookup + data download (open data) | 10 min, hourly, daily |
| **Agrometeo** | Agricultural weather network | Station lookup + data download | Hourly |

The `pyedautils` package provides a unified interface to both providers. This chapter shows how to:

1. Find the nearest weather station to any Swiss location
2. Download hourly weather data
3. Compare both station networks

```{note}
Both providers require an internet connection. All API calls go to public endpoints — no API key needed.
```

In [ ]:
# Colab setup: install pyedautils; elsewhere load plotly.js from a CDN for the plots
import sys
if "google.colab" in sys.modules:
    %pip install -q "pyedautils>=0.0.33"
else:
    import plotly.io as pio
    pio.renderers.default = "notebook_connected"

In [ ]:
import pandas as pd

(meteo-station-lookup)=
## Station Lookup

Both providers let you find the nearest weather station to a given location. The typical workflow is:

1. Convert a **postal code (PLZ)** to GPS coordinates
2. Find the **nearest station** that has the sensor you need

### Helper: PLZ to Coordinates

The `geopy` module converts any Swiss postal code to latitude/longitude and altitude.

In [ ]:
from pyedautils.geopy import get_coordindates_ch_plz, get_altitude_lat_long

plz = 6048  # Horw
coord = get_coordindates_ch_plz(plz)
altitude = get_altitude_lat_long(coord[0], coord[1])

print(f"PLZ {plz}: lat={coord[0]:.4f}, lon={coord[1]:.4f}, altitude={altitude:.0f} m")

### MeteoSwiss

MeteoSwiss operates Switzerland's official automatic weather stations. The `find_nearest_station()` function returns the **station abbreviation** (e.g. `"LUZ"` for Luzern) of the closest station that has the requested sensor and is within ±150 m altitude.

In [ ]:
from pyedautils.weather.meteo_swiss import find_nearest_station as find_meteoswiss

station_ms = find_meteoswiss(coord[0], coord[1], altitude, sensor="temp")
print(f"Nearest MeteoSwiss station for temperature: {station_ms}")

You can search for different sensor types: `"temp"`, `"globrad"` (global radiation), `"relhum"` (relative humidity), `"rain"`.

In [ ]:
for sensor in ["temp", "globrad", "relhum", "rain"]:
    station = find_meteoswiss(coord[0], coord[1], altitude, sensor=sensor)
    print(f"  {sensor:8s} → {station}")

### Agrometeo

[Agrometeo](https://www.agrometeo.ch) is a Swiss agricultural weather network with a public API for station lookup and data download.

In [ ]:
from pyedautils.weather.agroweather import find_nearest_station as find_agro

station_agro = find_agro(coord[0], coord[1], sensor="temp")
print(f"Nearest Agrometeo station for temperature: {station_agro}")

(meteo-station-overview)=
## Station Networks

Both providers expose a function to list all available stations. This is useful for understanding coverage and available sensors.

In [ ]:
from pyedautils.weather.meteo_swiss import get_current_station_data

stations_ms = get_current_station_data()
print(f"MeteoSwiss: {len(stations_ms)} stations")
stations_ms[["Abk.", "Station", "Stationshöhe m ü. M."]].head()

In [ ]:
from pyedautils.weather.agroweather import get_station_data

stations_agro = get_station_data()
print(f"Agrometeo: {len(stations_agro)} stations")
stations_agro[["id", "name", "lat", "lon", "altitude"]].head()

(meteo-download-data)=
## Downloading Weather Data

Both providers offer **hourly weather data** through public interfaces. MeteoSwiss publishes its measurements as [open data](https://opendatadocs.meteoswiss.ch/a-data-groundbased/a1-automatic-weather-stations); `download_data()` takes the station abbreviation and returns the data in UTC (or the time zone given with `tz`).

The examples use **fixed stations and a fixed past week**, so the results stay the same even if a new station opens closer to Horw. Here: MeteoSwiss station Luzern (`LUZ`) and Agrometeo station Meggen (ID `239`).

In [ ]:
from pyedautils.weather import meteo_swiss

df_luz = meteo_swiss.download_data("LUZ", "2026-06-01", "2026-06-07", tz="Europe/Zurich")
df_luz.head()

Agrometeo can be queried in two ways:

1. **By station ID** — if you already know the station
2. **By postal code (PLZ)** — automatically finds the nearest station(s)

### Download by Station ID

Use `download_data()` when you know the station ID, e.g. from `find_nearest_station()` above or from `get_station_data()`.

In [ ]:
from pyedautils.weather.agroweather import download_data

df_agro = download_data(
    station_id=239,  # Meggen, nearest Agrometeo station to Horw
    start_date="2026-06-01",
    end_date="2026-06-07",
    sensors=["temp", "globrad", "relhum", "rain"]
)

print(f"Shape: {df_agro.shape}")
print(f"Period: {df_agro.index.min()} to {df_agro.index.max()}")
df_agro.head()

### Download by Postal Code

The convenience function `download_data_by_plz()` handles the full workflow: PLZ → coordinates → nearest station(s) → download. If different sensors are closest to different stations, it automatically merges the results. Because the nearest station can change over time, use it for quick looks and pin the station ID for reproducible analyses:

```python
from pyedautils.weather.agroweather import download_data_by_plz

df_plz = download_data_by_plz(plz=6048, start_date="2026-06-01", end_date="2026-06-07")
```

In [ ]:
df_agro.describe()

### Quick Visualization

Let's plot the downloaded data to verify it looks reasonable.

In [ ]:
from pyedautils.plots import plot_timeseries

fig = plot_timeseries(
    df_agro,
    columns=["temp"],
    title="Outdoor Temperature — Meggen (Agrometeo 239)",
    ylab="Temperature [°C]",
)
fig.show()

In [ ]:
fig = plot_timeseries(
    df_agro,
    columns=["globrad"],
    title="Global Radiation — Meggen (Agrometeo 239)",
    ylab="Radiation [W/m²]",
)
fig.show()

## Available Sensors

Both providers support the same four sensor types:

| Sensor key | Description | Unit |
|---|---|---|
| `"temp"` | Air temperature | °C |
| `"globrad"` | Global radiation | W/m² |
| `"relhum"` | Relative humidity | % |
| `"rain"` | Precipitation | mm |

:::{admonition} Which provider should I use?
:class: tip dropdown

| Criteria | MeteoSwiss | Agrometeo |
|---|---|---|
| **Data download** | Yes, open data (10 min, hourly, daily, back to the start of the measurements) | Yes, via API (hourly) |
| **Altitude matching** | Yes (±150 m filter) | No |
| **Best for** | Official reference data, long time series | A station close to rural sites |

**Recommendation:** Start with the nearest **MeteoSwiss** station at a similar altitude; use **Agrometeo** when it is much closer to your building.
:::

## Summary

| Task | MeteoSwiss | Agrometeo |
|---|---|---|
| Find nearest station | `weather.meteo_swiss.find_nearest_station(lat, lon, alt, sensor)` | `weather.agroweather.find_nearest_station(lat, lon, sensor)` |
| List all stations | `weather.meteo_swiss.get_current_station_data()` | `weather.agroweather.get_station_data()` |
| Download data by station | `weather.meteo_swiss.download_data(station, start, end)` | `weather.agroweather.download_data(station_id, start, end)` |
| Download data by PLZ | `weather.meteo_swiss.download_data_by_plz(plz, start, end)` | `weather.agroweather.download_data_by_plz(plz, start, end)` |
| PLZ → coordinates | `geopy.get_coordindates_ch_plz(plz)` | (built into `download_data_by_plz`) |
| Get altitude | `geopy.get_altitude_lat_long(lat, lon)` | — |